# Resume NER: fine-tuning `dslim/bert-base-NER` on Colab GPU

Runtime → Change runtime type → **T4 GPU**. Upload the project folder as `resume-ner.zip` (without `.venv`, `models`, `frontend/node_modules`), or clone your repo. Add a Colab secret `KAGGLE_API_TOKEN` (Kaggle → Settings → API).

In [ ]:
from google.colab import files
up = files.upload()  # upload resume-ner.zip
!unzip -q -o resume-ner.zip -d /content/project
%cd /content/project

In [ ]:
!pip -q install transformers datasets accelerate seqeval scikit-learn pandas matplotlib seaborn pdfplumber python-docx kaggle

In [ ]:
import os
from google.colab import userdata
os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')
!kaggle datasets download -d dataturks/resume-entities-for-ner -p data/raw --unzip

## Phase 1 – data preparation and environment log

In [ ]:
!python src/data_prep.py
!python src/log_env.py

## Phase 2A – BEFORE: pretrained model as-is

In [ ]:
!python src/pretrained_eval.py

## Phase 2B – AFTER: hyperparameter grid (selected by validation F1), final run

In [ ]:
for bs in (8, 16):
    for lr in ('2e-5', '3e-5', '5e-5'):
        !python src/finetune.py --tag hp_lr{lr}_bs{bs} --lr {lr} --batch_size {bs} --epochs 10

In [ ]:
!python src/select_config.py

In [ ]:
import json
b = json.load(open('results/final_config.json'))['best']
ARGS = f"--lr {b['lr']} --batch_size {b['batch_size']} --epochs {b['epochs']}"
!python src/finetune.py --tag main {ARGS} --save_model

## Phase 3 – ablations and 5-fold CV

In [ ]:
!python src/finetune.py --tag abl_ep5 --lr {b['lr']} --batch_size {b['batch_size']} --epochs 5
!python src/finetune.py --tag abl_cw {ARGS} --class_weights
!python src/finetune.py --tag abl_os {ARGS} --oversample
!python src/finetune.py --tag abl_trunc {ARGS} --truncate
!python src/finetune.py --tag abl_raw {ARGS} --variant raw
!python src/finetune.py --tag cv_main {ARGS} --cv

## Phases 4–5 – evaluation, figures, error analysis

In [ ]:
!python src/benchmark.py   # note: Colab CPU timing differs from a laptop
!python src/evaluate.py
!python src/error_analysis.py

In [ ]:
!zip -qr outputs.zip results models/main
files.download('outputs.zip')